In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/shilpadevi1527/integrated-conversations-csv/integrated_conversations.csv


In [2]:
import pandas as pd

file_path = "/kaggle/input/datasets/shilpadevi1527/integrated-conversations-csv/integrated_conversations.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (276172, 8)

Columns:
['dialogue_id', 'turn_id', 'speaker', 'utterance', 'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin']


/tmp/ipykernel_58/1601900615.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


In [3]:
print(df.head(10).to_string())

                        dialogue_id  turn_id    speaker                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

In [4]:
print("\nSpeaker distribution:")
print(df["speaker"].value_counts())

print("\nSource distribution:")
print(df["source_dataset"].value_counts())

print("\nDialogue origin:")
print(df["dialogue_origin"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())


Speaker distribution:
speaker
user         138086
assistant    138086
Name: count, dtype: int64

Source distribution:
source_dataset
MedDialog              224328
HealthChat-LMSYS        28614
HealthChat-WildChat     23230
Name: count, dtype: int64

Dialogue origin:
dialogue_origin
constructed      224328
reconstructed     51844
Name: count, dtype: int64

Missing values:
dialogue_id             0
turn_id                 0
speaker                 0
utterance             198
source_dataset          0
original_id             0
source_label_raw    51844
dialogue_origin         0
dtype: int64


In [5]:
missing_rows = df[df["utterance"].isna()]

print("Number of missing utterances:", len(missing_rows))
print("\nMissing utterances by speaker:")
print(missing_rows["speaker"].value_counts())

print("\nMissing utterances by source:")
print(missing_rows["source_dataset"].value_counts())

print("\nMissing utterance rows:")
print(missing_rows[
    ["dialogue_id", "turn_id", "speaker", "source_dataset", "original_id"]
].to_string(index=False))

Number of missing utterances: 198

Missing utterances by speaker:
speaker
assistant    149
user          49
Name: count, dtype: int64

Missing utterances by source:
source_dataset
HealthChat-LMSYS       149
HealthChat-WildChat     49
Name: count, dtype: int64

Missing utterance rows:
                     dialogue_id  turn_id   speaker      source_dataset                      original_id
00801e92edad4f9eba6d8558c284553d       11 assistant    HealthChat-LMSYS 00801e92edad4f9eba6d8558c284553d
0190c13200ac4db0bfcb1a278de1f1a2        7 assistant    HealthChat-LMSYS 0190c13200ac4db0bfcb1a278de1f1a2
0190c13200ac4db0bfcb1a278de1f1a2        9 assistant    HealthChat-LMSYS 0190c13200ac4db0bfcb1a278de1f1a2
0473eeb47eb543a39216a3d728b36865       11 assistant    HealthChat-LMSYS 0473eeb47eb543a39216a3d728b36865
06ba3a0e654745d190448be10f7617e8        9 assistant    HealthChat-LMSYS 06ba3a0e654745d190448be10f7617e8
07326d11c659430c801e91bb0c24ff4a       15 assistant    HealthChat-LMSYS 07326d11c6594

In [6]:
# Conversations that contain at least one missing utterance
affected_dialogues = df.loc[
    df["utterance"].isna(),
    "dialogue_id"
].unique()

print("Number of affected conversations:", len(affected_dialogues))

# Show how many missing turns each affected conversation has
missing_by_dialogue = (
    df[df["dialogue_id"].isin(affected_dialogues)]
    .groupby(["dialogue_id", "source_dataset"])["utterance"]
    .apply(lambda x: x.isna().sum())
    .reset_index(name="missing_turns")
)

print("\nMissing turns per affected conversation:")
print(missing_by_dialogue.to_string(index=False))

Number of affected conversations: 139

Missing turns per affected conversation:
                     dialogue_id      source_dataset  missing_turns
00801e92edad4f9eba6d8558c284553d    HealthChat-LMSYS              1
013c533871805e16bda5c173f7719c0a HealthChat-WildChat              1
0190c13200ac4db0bfcb1a278de1f1a2    HealthChat-LMSYS              2
0473eeb47eb543a39216a3d728b36865    HealthChat-LMSYS              1
04b4d6e1166144114cae072a7ad1b6a9 HealthChat-WildChat              1
066839ca3b3da4ec85915539029ff39f HealthChat-WildChat              1
06ba3a0e654745d190448be10f7617e8    HealthChat-LMSYS              1
07326d11c659430c801e91bb0c24ff4a    HealthChat-LMSYS              1
0732f41ebbbf4de19573f2f64785e9ee    HealthChat-LMSYS              1
077749bd4cdb4d50b1d1645c67b7c88f    HealthChat-LMSYS              7
07fdaea703574573bd86de7cdb48066e    HealthChat-LMSYS              1
08af95bb551b41638d3e6419d44ca48e    HealthChat-LMSYS              1
0d6bb1e672d94f43beabb91815b1b445    

In [7]:
# Show the first 5 affected conversations with their surrounding turns

for dialogue_id in affected_dialogues[:5]:
    print("\n" + "=" * 80)
    print("DIALOGUE:", dialogue_id)

    temp = df[df["dialogue_id"] == dialogue_id].sort_values("turn_id")

    print(temp[[
        "turn_id",
        "speaker",
        "utterance",
        "source_dataset"
    ]].to_string(index=False))


DIALOGUE: 00801e92edad4f9eba6d8558c284553d
 turn_id   speaker                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [9]:
# Remove rows where the utterance is missing
df_clean = df.dropna(subset=["utterance"]).copy()

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("\nRemaining missing utterances:")
print(df_clean["utterance"].isna().sum())

Original shape: (276172, 8)
Cleaned shape: (275974, 8)

Remaining missing utterances:
0


In [10]:
# Check for empty or whitespace-only utterances
empty_utterances = df_clean["utterance"].astype(str).str.strip().eq("").sum()

print("Empty/blank utterances:", empty_utterances)

Empty/blank utterances: 0


In [11]:
# Check number of turns per conversation
turns_per_dialogue = df_clean.groupby("dialogue_id").size()

print("Total dialogues:", turns_per_dialogue.shape[0])
print("\nTurn count distribution:")
print(turns_per_dialogue.value_counts().sort_index())

print("\nMinimum turns in a dialogue:", turns_per_dialogue.min())
print("Maximum turns in a dialogue:", turns_per_dialogue.max())

Total dialogues: 123256

Turn count distribution:
1           1
2      118357
3           2
4        1958
5          14
6        1142
7           8
8         518
9          10
10        345
11         16
12        231
13         16
14        152
15         17
16        103
17         10
18         91
19          4
20         60
21          5
22         39
23          1
24         25
25          1
26         21
28         28
30         12
32         11
34          7
36          9
38         10
40          6
42          1
43          1
44          2
46          4
47          1
48          1
50          1
52          1
53          1
54          1
56          1
58          2
60          2
100         7
Name: count, dtype: int64

Minimum turns in a dialogue: 1
Maximum turns in a dialogue: 100


In [12]:
# Check whether speakers alternate correctly within each dialogue

invalid_dialogues = []

for dialogue_id, group in df_clean.groupby("dialogue_id"):
    speakers = group.sort_values("turn_id")["speaker"].tolist()

    for i in range(1, len(speakers)):
        if speakers[i] == speakers[i - 1]:
            invalid_dialogues.append(dialogue_id)
            break

print("Dialogues with consecutive same speakers:", len(set(invalid_dialogues)))

Dialogues with consecutive same speakers: 71


In [14]:
# Check consecutive speakers without looping through every dialogue

df_check = df_clean.sort_values(["dialogue_id", "turn_id"]).copy()

same_as_previous = (
    df_check["speaker"] == df_check.groupby("dialogue_id")["speaker"].shift(1)
)

affected_dialogues = df_check.loc[
    same_as_previous, "dialogue_id"
].unique()

print("Dialogues with consecutive same speakers:", len(affected_dialogues))

Dialogues with consecutive same speakers: 71


In [15]:
# Show the affected dialogues
for dialogue_id in affected_dialogues[:10]:
    temp = df_check[df_check["dialogue_id"] == dialogue_id]

    print("\n" + "=" * 80)
    print("Dialogue ID:", dialogue_id)
    print(temp[["turn_id", "speaker", "utterance", "source_dataset"]].to_string(index=False))


Dialogue ID: 013c533871805e16bda5c173f7719c0a
 turn_id   speaker                                                                                                                                                                                                                                                                                                                                                                                        utterance      source_dataset
       0      user                                                                                                                                                                                                                                                                                                                                        Hydrogen peroxide mouth wash is what type of preparation  HealthChat-WildChat
       1 assistant                                                                                       

In [16]:
# Check for gaps in turn IDs within each dialogue

turn_counts = (
    df_clean.groupby("dialogue_id")["turn_id"]
    .agg(["min", "max", "count"])
)

turn_counts["expected_count"] = (
    turn_counts["max"] - turn_counts["min"] + 1
)

gapped_dialogues = turn_counts[
    turn_counts["count"] != turn_counts["expected_count"]
]

print("Dialogues with turn_id gaps:", len(gapped_dialogues))
print("Total dialogues:", len(turn_counts))

Dialogues with turn_id gaps: 71
Total dialogues: 123256


In [17]:
# Create the final master dataset for NLU preparation

df_master = df_clean.copy()

print("Master dataset shape:", df_master.shape)
print("Missing utterances:", df_master["utterance"].isna().sum())
print("Empty utterances:",
      df_master["utterance"].astype(str).str.strip().eq("").sum())

Master dataset shape: (275974, 8)
Missing utterances: 0
Empty utterances: 0


In [18]:
# Basic text statistics for NLU preparation

df_master["text_length"] = df_master["utterance"].astype(str).str.len()

print("Average characters:", round(df_master["text_length"].mean(), 2))
print("Median characters:", df_master["text_length"].median())
print("Maximum characters:", df_master["text_length"].max())

print("\nTurns by speaker:")
print(df_master["speaker"].value_counts())

print("\nTurns by source:")
print(df_master["source_dataset"].value_counts())

Average characters: 578.18
Median characters: 479.0
Maximum characters: 40404

Turns by speaker:
speaker
user         138037
assistant    137937
Name: count, dtype: int64

Turns by source:
source_dataset
MedDialog              224328
HealthChat-LMSYS        28465
HealthChat-WildChat     23181
Name: count, dtype: int64


In [19]:
# Check how many utterances are very long

for limit in [1000, 2000, 4000, 8000, 16000]:
    count = (df_master["text_length"] > limit).sum()
    print(f"Utterances longer than {limit} characters: {count}")

Utterances longer than 1000 characters: 28050
Utterances longer than 2000 characters: 5371
Utterances longer than 4000 characters: 297
Utterances longer than 8000 characters: 29
Utterances longer than 16000 characters: 9


In [20]:
# Remove the temporary text_length column

df_master = df_master.drop(columns=["text_length"])

print("Columns:")
print(df_master.columns.tolist())

print("\nShape:", df_master.shape)

Columns:
['dialogue_id', 'turn_id', 'speaker', 'utterance', 'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin']

Shape: (275974, 8)


In [22]:
# Display one complete conversation

sample_dialogue_id = df_master["dialogue_id"].iloc[0]

sample = (
    df_master[df_master["dialogue_id"] == sample_dialogue_id]
    .sort_values("turn_id")
)

print("Dialogue ID:", sample_dialogue_id)
print("Number of turns:", len(sample))

for _, row in sample.iterrows():
    print(f"\n{row['speaker'].upper()}:")
    print(row["utterance"])

Dialogue ID: 000573958699464e9de6493b5e182fab
Number of turns: 10

USER:
Prompt: I want you to be my personal mental health Al counselor. I am a 16 year old male, struggling with OCD. Let's have a counseling session. I want you to utilize evidence-based therapeutic techniques to guide me through my struggles and help me improve my mental health.  Remember we are having a chat -- don't overwhelm me with a lot of text and keep the conversation flowing. Input: "Hey there, I'm NAME_1"

ASSISTANT:
NAME_2, it's nice to meet you. I'm here to help you with your mental health, specifically with your OCD. Let's start by talking about what you're currently experiencing with your OCD. What are some of the thoughts or behaviors that are causing you distress?

USER:
I guess there's quite a few things. But I primarily struggle with cleanliness 

ASSISTANT:
Okay, that's a common theme for many people with OCD. Can you tell me more about what you do to try and maintain cleanliness? Are there certain th

In [23]:
# Count dialogues by source dataset

dialogue_sources = (
    df_master.groupby(["dialogue_id", "source_dataset"])
    .size()
    .reset_index(name="turn_count")
)

print(dialogue_sources["source_dataset"].value_counts())

print("\nNumber of dialogues by source:")
print(
    dialogue_sources
    .groupby("source_dataset")["dialogue_id"]
    .nunique()
)

source_dataset
MedDialog              112164
HealthChat-LMSYS         6828
HealthChat-WildChat      4264
Name: count, dtype: int64

Number of dialogues by source:
source_dataset
HealthChat-LMSYS         6828
HealthChat-WildChat      4264
MedDialog              112164
Name: dialogue_id, dtype: int64


In [24]:
# Inspect the available labels in the cleaned dataset

print("Total non-null source labels:",
      df_master["source_label_raw"].notna().sum())

print("\nUnique source labels:")
print(df_master["source_label_raw"].dropna().value_counts().head(30))

print("\nNumber of unique labels:",
      df_master["source_label_raw"].nunique())

Total non-null source labels: 224328

Unique source labels:
source_label_raw
If you are a doctor, please answer the medical questions based on the patient's description.    224328
Name: count, dtype: int64

Number of unique labels: 1


In [25]:
# Create conversation-level training examples

def build_conversation(group):
    group = group.sort_values("turn_id")

    conversation = []

    for _, row in group.iterrows():
        speaker = row["speaker"].capitalize()
        utterance = str(row["utterance"]).strip()

        conversation.append(f"{speaker}: {utterance}")

    return "\n".join(conversation)


df_conversations = (
    df_master
    .groupby("dialogue_id", sort=False)
    .apply(build_conversation)
    .reset_index(name="text")
)

print("Conversation dataset shape:", df_conversations.shape)

print("\nColumns:")
print(df_conversations.columns.tolist())

print("\nFirst conversation:")
print(df_conversations.iloc[0]["text"])

Conversation dataset shape: (123256, 2)

Columns:
['dialogue_id', 'text']

First conversation:
User: Prompt: I want you to be my personal mental health Al counselor. I am a 16 year old male, struggling with OCD. Let's have a counseling session. I want you to utilize evidence-based therapeutic techniques to guide me through my struggles and help me improve my mental health.  Remember we are having a chat -- don't overwhelm me with a lot of text and keep the conversation flowing. Input: "Hey there, I'm NAME_1"
Assistant: NAME_2, it's nice to meet you. I'm here to help you with your mental health, specifically with your OCD. Let's start by talking about what you're currently experiencing with your OCD. What are some of the thoughts or behaviors that are causing you distress?
User: I guess there's quite a few things. But I primarily struggle with cleanliness
Assistant: Okay, that's a common theme for many people with OCD. Can you tell me more about what you do to try and maintain cleanline

/tmp/ipykernel_58/2957407992.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_conversation)


In [26]:
# Recreate conversation dataset without the pandas warning

df_conversations = (
    df_master
    .groupby("dialogue_id", sort=False)[["dialogue_id", "turn_id", "speaker", "utterance"]]
    .apply(
        lambda group: "\n".join(
            f"{row['speaker'].capitalize()}: {str(row['utterance']).strip()}"
            for _, row in group.sort_values("turn_id").iterrows()
        )
    )
    .reset_index(name="text")
)

print("Conversation dataset shape:", df_conversations.shape)

print("\nMissing conversations:")
print(df_conversations["text"].isna().sum())

print("\nFirst conversation preview:")
print(df_conversations.iloc[0]["text"][:1000])

Conversation dataset shape: (123256, 2)

Missing conversations:
0

First conversation preview:
User: Prompt: I want you to be my personal mental health Al counselor. I am a 16 year old male, struggling with OCD. Let's have a counseling session. I want you to utilize evidence-based therapeutic techniques to guide me through my struggles and help me improve my mental health.  Remember we are having a chat -- don't overwhelm me with a lot of text and keep the conversation flowing. Input: "Hey there, I'm NAME_1"
Assistant: NAME_2, it's nice to meet you. I'm here to help you with your mental health, specifically with your OCD. Let's start by talking about what you're currently experiencing with your OCD. What are some of the thoughts or behaviors that are causing you distress?
User: I guess there's quite a few things. But I primarily struggle with cleanliness
Assistant: Okay, that's a common theme for many people with OCD. Can you tell me more about what you do to try and maintain cleanline

In [27]:
from sklearn.model_selection import train_test_split

# Split at conversation level
train_df, val_df = train_test_split(
    df_conversations,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

# Reset indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training conversations:", len(train_df))
print("Validation conversations:", len(val_df))

print("\nTraining percentage:",
      round(len(train_df) / len(df_conversations) * 100, 2), "%")

print("Validation percentage:",
      round(len(val_df) / len(df_conversations) * 100, 2), "%")

Training conversations: 110930
Validation conversations: 12326

Training percentage: 90.0 %
Validation percentage: 10.0 %


In [28]:
# Verify that no dialogue appears in both train and validation

train_ids = set(train_df["dialogue_id"])
val_ids = set(val_df["dialogue_id"])

overlap = train_ids.intersection(val_ids)

print("Training dialogue IDs:", len(train_ids))
print("Validation dialogue IDs:", len(val_ids))
print("Overlapping dialogue IDs:", len(overlap))

Training dialogue IDs: 110930
Validation dialogue IDs: 12326
Overlapping dialogue IDs: 0


In [29]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 77.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.1 MB/s eta 0:00:00:00:01


In [30]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("⚠️ No GPU detected")

PyTorch version: 2.10.0+cpu
CUDA available: False
⚠️ No GPU detected


In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("⚠️ No GPU detected")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
import transformers
import datasets
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

ModuleNotFoundError: No module named 'bitsandbytes'

In [4]:
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 37.7 MB/s eta 0:00:00:00:0100:01


In [5]:
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.2


In [6]:
from transformers import AutoTokenizer

model_name = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 32000


In [7]:
# Tokenize a sample of our training conversations

sample_texts = train_df["text"].sample(
    n=min(1000, len(train_df)),
    random_state=42
).tolist()

token_lengths = [
    len(tokenizer(text, add_special_tokens=True)["input_ids"])
    for text in sample_texts
]

import numpy as np

print("Number of conversations checked:", len(token_lengths))
print("Minimum tokens:", min(token_lengths))
print("Median tokens:", int(np.median(token_lengths)))
print("Average tokens:", int(np.mean(token_lengths)))
print("90th percentile:", int(np.percentile(token_lengths, 90)))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Maximum tokens:", max(token_lengths))

NameError: name 'train_df' is not defined

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split

file_path = "/kaggle/input/datasets/shilpadevi1527/integrated-conversations-csv/integrated_conversations.csv"

# Load dataset
df = pd.read_csv(file_path)

# Remove rows with missing utterances
df_master = df.dropna(subset=["utterance"]).copy()

# Build one conversation per dialogue
df_conversations = (
    df_master
    .groupby("dialogue_id", sort=False)[
        ["dialogue_id", "turn_id", "speaker", "utterance"]
    ]
    .apply(
        lambda group: "\n".join(
            f"{row['speaker'].capitalize()}: {str(row['utterance']).strip()}"
            for _, row in group.sort_values("turn_id").iterrows()
        )
    )
    .reset_index(name="text")
)

# Same 90/10 split as before
train_df, val_df = train_test_split(
    df_conversations,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Total conversations:", len(df_conversations))
print("Training conversations:", len(train_df))
print("Validation conversations:", len(val_df))

Total conversations: 123256
Training conversations: 110930
Validation conversations: 12326


In [15]:
import numpy as np

sample_texts = train_df["text"].sample(
    n=min(1000, len(train_df)),
    random_state=42
).tolist()

token_lengths = [
    len(tokenizer(text, add_special_tokens=True)["input_ids"])
    for text in sample_texts
]

print("Number of conversations checked:", len(token_lengths))
print("Minimum tokens:", min(token_lengths))
print("Median tokens:", int(np.median(token_lengths)))
print("Average tokens:", int(np.mean(token_lengths)))
print("90th percentile:", int(np.percentile(token_lengths, 90)))
print("95th percentile:", int(np.percentile(token_lengths, 95)))
print("Maximum tokens:", max(token_lengths))

Number of conversations checked: 1000
Minimum tokens: 128
Median tokens: 258
Average tokens: 315
90th percentile: 410
95th percentile: 495
Maximum tokens: 4776


In [16]:
from datasets import Dataset

# Keep only the conversation text
train_dataset = Dataset.from_pandas(
    train_df[["text"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text"]],
    preserve_index=False
)

print("Training dataset:", train_dataset)
print("Validation dataset:", val_dataset)

print("\nExample:")
print(train_dataset[0]["text"][:500])

Training dataset: Dataset({
    features: ['text'],
    num_rows: 110930
})
Validation dataset: Dataset({
    features: ['text'],
    num_rows: 12326
})

Example:
User: What are the effects of the drug buspirone and how does it work biologically?
Assistant: Buspirone is a medication that is primarily used to treat anxiety disorders. It works by modulating the activity of neurotransmitters in the brain, specifically serotonin and dopamine. Buspirone acts as a serotonin 1A receptor agonist, which means that it binds to and stimulates these receptors, leading to an increase in the levels of serotonin in the brain. This increased serotonin activity can help t


In [17]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training data"
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing validation data"
)

print("Training tokenized:", train_tokenized)
print("Validation tokenized:", val_tokenized)

Tokenizing training data:   0%|          | 0/110930 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/12326 [00:00<?, ? examples/s]

Training tokenized: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 110930
})
Validation tokenized: Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 12326
})


In [18]:
print("Number of tokens:", len(train_tokenized[0]["input_ids"]))
print("Number of attention values:", len(train_tokenized[0]["attention_mask"]))

print("\nFirst 20 token IDs:")
print(train_tokenized[0]["input_ids"][:20])

print("\nAttention mask:")
print(train_tokenized[0]["attention_mask"][:20])

Number of tokens: 287
Number of attention values: 287

First 20 token IDs:
[1, 1247, 28747, 1824, 460, 272, 6092, 302, 272, 7876, 1579, 28720, 361, 538, 304, 910, 1235, 378, 771, 4240]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [19]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_name = "BioMistral/BioMistral-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("BioMistral-7B loaded successfully!")
print("Model device:", model.device)

pytorch_model.bin:   0%|          | 0.00/14.5G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')
Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 76, in get_conversion_pr_reference
    raise OSError(
OSError: Could not create safetensors conversion PR. The repo does no

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

BioMistral-7B loaded successfully!
Model device: cuda:0


In [20]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


In [21]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./biomistral_qlora",
    
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    
    learning_rate=2e-4,
    weight_decay=0.01,
    
    fp16=True,
    gradient_checkpointing=True,
    
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    
    report_to="none",
    optim="paged_adamw_8bit",
    
    remove_unused_columns=False
)

print("Training configuration created successfully!")

Training configuration created successfully!


In [23]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator created successfully!")

Data collator created successfully!


In [24]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Trainer created successfully!")

Trainer created successfully!


In [25]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory before training:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

# Temporarily train on only 10 examples
test_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized.select(range(10)),
    data_collator=data_collator,
)

test_result = test_trainer.train()

print("\nSanity-check training completed!")
print("Training loss:", test_result.training_loss)

print("GPU memory after training:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

GPU: Tesla T4
GPU memory before training: 1.77 GB


ValueError: You have set `args.eval_strategy` to IntervalStrategy.STEPS but you didn't pass an `eval_dataset` to `Trainer`. Either set `args.eval_strategy` to `no` or pass an `eval_dataset`. 

In [26]:
import torch
from transformers import TrainingArguments, Trainer

test_args = TrainingArguments(
    output_dir="./biomistral_test",
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False
)

test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=train_tokenized.select(range(10)),
    data_collator=data_collator,
)

print("Starting 10-step sanity check...")

test_result = test_trainer.train()

print("\nSanity-check training completed!")
print("Training loss:", test_result.training_loss)

print(
    "GPU memory after training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Starting 10-step sanity check...


ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

In [27]:
tokenizer.pad_token = tokenizer.eos_token

print("PAD token:", tokenizer.pad_token)
print("PAD token ID:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)

PAD token: </s>
PAD token ID: 2
EOS token: </s>
EOS token ID: 2


In [28]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator updated successfully!")

Data collator updated successfully!


In [29]:
import torch
from transformers import TrainingArguments, Trainer

test_args = TrainingArguments(
    output_dir="./biomistral_test",
    max_steps=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False
)

test_trainer = Trainer(
    model=model,
    args=test_args,
    train_dataset=train_tokenized.select(range(10)),
    data_collator=data_collator,
)

print("Starting 10-step sanity check...")

test_result = test_trainer.train()

print("\nSanity-check training completed!")
print("Training loss:", test_result.training_loss)

print(
    "GPU memory after training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Starting 10-step sanity check...


Step,Training Loss
1,2.692143
2,2.811722
3,2.611493
4,2.801223
5,2.404019
6,2.963716
7,0.943500
8,2.357422
9,2.952361
10,3.018791



Sanity-check training completed!
Training loss: 2.5556390225887298
GPU memory after training: 1.79 GB


In [30]:
import gc
import torch

del test_trainer
del test_result

gc.collect()
torch.cuda.empty_cache()

print("Temporary test trainer cleared.")
print(
    "GPU memory currently allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Temporary test trainer cleared.
GPU memory currently allocated: 1.78 GB


In [31]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Real Trainer recreated successfully!")
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))

Real Trainer recreated successfully!
Training examples: 110930
Validation examples: 12326


In [32]:
print("Starting BioMistral-7B QLoRA training...")
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(val_tokenized))

train_result = trainer.train()

print("\nTraining completed!")
print("Final training loss:", train_result.training_loss)

Starting BioMistral-7B QLoRA training...
Training examples: 110930
Validation examples: 12326


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [33]:
# Use a reproducible 20,000-conversation subset
TRAIN_SIZE = 20_000

train_small = train_tokenized.shuffle(seed=42).select(range(TRAIN_SIZE))

print("Training examples:", len(train_small))
print("Validation examples:", len(val_tokenized))

Training examples: 20000
Validation examples: 12326


In [35]:
from transformers import TrainingArguments

training_args_small = TrainingArguments(
    output_dir="./biomistral_qlora_20k",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,

    fp16=True,
    gradient_checkpointing=True,

    logging_steps=25,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    report_to="none",
    optim="paged_adamw_8bit",

    remove_unused_columns=False
)

print("20K training configuration created!")

20K training configuration created!


In [36]:
from transformers import Trainer

trainer_small = Trainer(
    model=model,
    args=training_args_small,
    train_dataset=train_small,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("20K Trainer created successfully!")
print("Training examples:", len(train_small))
print("Validation examples:", len(val_tokenized))

20K Trainer created successfully!
Training examples: 20000
Validation examples: 12326


In [37]:
import time

speed_args = TrainingArguments(
    output_dir="./biomistral_speed_test",
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    fp16=True,
    gradient_checkpointing=True,

    logging_steps=25,
    eval_strategy="no",
    save_strategy="no",

    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False
)

speed_trainer = Trainer(
    model=model,
    args=speed_args,
    train_dataset=train_small,
    data_collator=data_collator,
)

print("Starting 100-step speed test...")
start_time = time.time()

speed_result = speed_trainer.train()

elapsed = time.time() - start_time

print("\n100-step test completed!")
print("Time taken:", round(elapsed / 60, 2), "minutes")
print("Average time per step:", round(elapsed / 100, 2), "seconds")
print("Training loss:", speed_result.training_loss)

Starting 100-step speed test...


CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 86
Number of tensors saved during recomputation: 42.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


In [38]:
# Disable gradient checkpointing to avoid the PyTorch checkpoint recomputation error

model.gradient_checkpointing_disable()

print("Gradient checkpointing:", model.is_gradient_checkpointing)

Gradient checkpointing: False


In [39]:
import time
from transformers import TrainingArguments, Trainer

speed_args = TrainingArguments(
    output_dir="./biomistral_speed_test",
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    fp16=True,
    gradient_checkpointing=False,

    logging_steps=25,
    eval_strategy="no",
    save_strategy="no",

    report_to="none",
    optim="paged_adamw_8bit",
    remove_unused_columns=False
)

speed_trainer = Trainer(
    model=model,
    args=speed_args,
    train_dataset=train_small,
    data_collator=data_collator,
)

print("Starting 100-step speed test...")
start_time = time.time()

speed_result = speed_trainer.train()

elapsed = time.time() - start_time

print("\n100-step test completed!")
print("Time taken:", round(elapsed / 60, 2), "minutes")
print("Average time per step:", round(elapsed / 100, 2), "seconds")
print("Training loss:", speed_result.training_loss)


Starting 100-step speed test...


Step,Training Loss
25,2.258837
50,2.249274
75,2.219222
100,2.199537



100-step test completed!
Time taken: 15.69 minutes
Average time per step: 9.42 seconds
Training loss: 2.2317174911499023


In [40]:
from transformers import TrainingArguments

final_args = TrainingArguments(
    output_dir="./biomistral_qlora_20k",

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    weight_decay=0.01,

    fp16=True,
    gradient_checkpointing=False,

    logging_steps=100,

    eval_strategy="no",

    save_strategy="epoch",
    save_total_limit=1,

    report_to="none",
    optim="paged_adamw_8bit",

    remove_unused_columns=False
)

print("Final 20K training configuration ready!")

Final 20K training configuration ready!


In [41]:
from transformers import Trainer

final_trainer = Trainer(
    model=model,
    args=final_args,
    train_dataset=train_small,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
)

print("Final Trainer created successfully!")
print("Training examples:", len(train_small))
print("Validation examples:", len(val_tokenized))

Final Trainer created successfully!
Training examples: 20000
Validation examples: 12326


In [43]:
print("🚀 Starting BioMistral-7B QLoRA training on 20,000 conversations...")
print("Training examples:", len(train_small))

train_result = final_trainer.train()

print("\n✅ Training completed!")
print("Final training loss:", train_result.training_loss)

🚀 Starting BioMistral-7B QLoRA training on 20,000 conversations...
Training examples: 20000


KeyboardInterrupt: 

In [44]:
print("Model:", model_name)
print("Model loaded on:", model.device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("LoRA attached:", hasattr(model, "peft_config"))

Model: BioMistral/BioMistral-7B
Model loaded on: cuda:0
Parameters: 3765702656
LoRA attached: True


In [45]:
model = model.unload()

print("LoRA removed successfully!")
print("Model:", model_name)
print("Device:", model.device)

LoRA removed successfully!
Model: BioMistral/BioMistral-7B
Device: cuda:0


In [47]:
import torch

question = "What are the common symptoms of iron deficiency anemia?"

prompt = f"User: {question}\nAssistant:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("INPUT:")
print(question)

print("\nMODEL OUTPUT:")
print(response)

INPUT:
What are the common symptoms of iron deficiency anemia?

MODEL OUTPUT:
User: What are the common symptoms of iron deficiency anemia?
Assistant: The common symptoms of iron deficiency anemia include fatigue, weakness, paleness, shortness of breath, chest pain, dizziness, headaches, cold extremities, and dark circles under the eyes.
